# Companion Code: Boolean Fourier Analysis and the Walsh–Hadamard Transform

This notebook provides runnable code for the key computations in the guide.
Each section corresponds to a chapter. Where possible, we give both a
**NumPy** implementation (clear, pedagogical) and a **PyTorch** implementation
(GPU-ready, relevant to the thesis's ML setting).

**Requirements:** `numpy`, `torch`, `matplotlib`, `scipy`, `sympy` — see `requirements.txt`.

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
from itertools import combinations
from math import comb, factorial, sqrt, pi
from typing import Callable

np.set_printoptions(precision=6, suppress=True)

## §0. Utilities

Core building blocks used throughout: the Fast Walsh–Hadamard Transform,
hypercube enumeration, and Fourier coefficient computation.

In [ ]:
# ── Fast Walsh–Hadamard Transform ──────────────────────────────────────
# In-place butterfly algorithm (Algorithm 5.1 in the guide).
# Computes H_n @ v (unnormalized). Divide by N for Fourier coefficients.

def fwht_numpy(v: np.ndarray) -> np.ndarray:
    """Unnormalized FWHT.  Returns H_n @ v (a fresh copy)."""
    a = v.astype(float).copy()
    N = len(a)
    h = 1
    while h < N:
        for j in range(0, N, 2 * h):
            for i in range(j, j + h):
                x, y = a[i], a[i + h]
                a[i], a[i + h] = x + y, x - y
        h *= 2
    return a


def fwht_torch(v: torch.Tensor) -> torch.Tensor:
    """Unnormalized FWHT (vectorized, no Python loops over elements)."""
    a = v.clone().float()
    N = a.shape[-1]
    h = 1
    while h < N:
        # reshape into blocks of size 2h, then butterfly
        a_view = a.view(-1, N // (2 * h), 2, h)
        top = a_view[:, :, 0, :]
        bot = a_view[:, :, 1, :]
        s, d = top + bot, top - bot
        a_view[:, :, 0, :] = s
        a_view[:, :, 1, :] = d
        h *= 2
    return a.view_as(v)


# Quick sanity check: H_2 @ (1, -1, -1, 1) should be (0, 0, 0, 4)
assert np.allclose(fwht_numpy(np.array([1, -1, -1, 1])), [0, 0, 0, 4])
v_t = torch.tensor([[1, -1, -1, 1]])  # batch dim for torch version
assert torch.allclose(fwht_torch(v_t), torch.tensor([[0, 0, 0, 4]], dtype=torch.float32))
print("FWHT implementations OK")

In [ ]:
# ── Hypercube helpers ──────────────────────────────────────────────────

def hypercube_points(n: int) -> np.ndarray:
    """All 2^n points of {±1}^n in binary-counting order.

    Column c -> x = ((-1)^{c_0}, (-1)^{c_1}, ..., (-1)^{c_{n-1}}).
    This matches the convention in the guide (Remark 5.2).
    """
    N = 1 << n
    points = np.empty((N, n), dtype=int)
    for c in range(N):
        for i in range(n):
            points[c, i] = 1 - 2 * ((c >> i) & 1)  # (-1)^{c_i}
    return points


def subset_to_index(S: set, n: int) -> int:
    """Subset S ⊆ [n] -> row index r (binary representation)."""
    r = 0
    for i in S:
        r |= (1 << (i - 1))  # bit (i-1) set for element i ∈ S
    return r


def index_to_subset(r: int, n: int) -> set:
    """Row index r -> subset S ⊆ [n]."""
    return {i + 1 for i in range(n) if (r >> i) & 1}


def all_subsets_up_to_degree(n: int, k: int):
    """Yield all S ⊆ [n] with |S| ≤ k, sorted by size then lex."""
    for size in range(k + 1):
        for S in combinations(range(1, n + 1), size):
            yield set(S)


# ── Fourier coefficient computation ───────────────────────────────────

def fourier_coefficients(f_values: np.ndarray) -> np.ndarray:
    """Compute all Fourier coefficients via normalized FWHT.

    f_values[c] = f(x_c) where x_c is the c-th hypercube point
    in binary-counting order.
    Returns array where entry r = f̂(S_r).
    """
    N = len(f_values)
    return fwht_numpy(f_values) / N


def evaluate_from_coefficients(f_hat: np.ndarray) -> np.ndarray:
    """Reconstruct function values from Fourier coefficients via FWHT.

    Inverse of fourier_coefficients: f_values = H_n @ f_hat.
    """
    return fwht_numpy(f_hat)


# ── Standard Boolean functions ────────────────────────────────────────

def make_and(n: int) -> np.ndarray:
    """AND_n: f(x) = 1 iff all x_i = 1, else -1."""
    pts = hypercube_points(n)
    return np.where(np.all(pts == 1, axis=1), 1, -1).astype(float)


def make_majority(n: int) -> np.ndarray:
    """Maj_n: f(x) = sgn(x_1 + ... + x_n).  Requires n odd."""
    assert n % 2 == 1, "Majority needs odd n"
    pts = hypercube_points(n)
    return np.sign(pts.sum(axis=1)).astype(float)


def make_parity(n: int) -> np.ndarray:
    """Parity (χ_{[n]}): f(x) = x_1 · x_2 · ... · x_n."""
    pts = hypercube_points(n)
    return pts.prod(axis=1).astype(float)


def make_dictator(n: int, i: int = 1) -> np.ndarray:
    """Dictator on coordinate i: f(x) = x_i."""
    pts = hypercube_points(n)
    return pts[:, i - 1].astype(float)


print("Helpers loaded")

---
## §1. Ch2 — Boolean Fourier Basics

Verify the worked examples: Fourier coefficients of AND, Maj₃, parity.
Check Parseval. Demonstrate truncation and sign-rounding.

In [ ]:
# ── Example 2.2: Fourier coefficients of AND_2 ───────────────────────
n = 2
f = make_and(n)
f_hat = fourier_coefficients(f)

print("AND_2 function values:", f)
print("AND_2 Fourier coefficients:")
for r in range(1 << n):
    S = index_to_subset(r, n)
    label = str(S) if S else "∅"
    print(f"  f̂({label:>6s}) = {f_hat[r]:+.4f}")

# Expected from the guide (Example 2.2):
#   f̂(∅) = -1/2,  f̂({1}) = 1/2,  f̂({2}) = 1/2,  f̂({1,2}) = 1/2
expected = np.array([-1/2, 1/2, 1/2, 1/2])
assert np.allclose(f_hat, expected), f"Mismatch: {f_hat} vs {expected}"
print("\n✓ Matches Example 2.2")

In [ ]:
# ── Example 2.3: Fourier coefficients of Maj_3 ──────────────────────
n = 3
f = make_majority(n)
f_hat = fourier_coefficients(f)

print("Maj_3 function values:", f)
print("Maj_3 Fourier coefficients:")
for r in range(1 << n):
    S = index_to_subset(r, n)
    label = str(S) if S else "∅"
    print(f"  f̂({label:>10s}) = {f_hat[r]:+.4f}")

# Expected (Example 2.3): f̂(∅)=0, f̂({i})=1/2, f̂({i,j})=0, f̂({1,2,3})=-1/2
print("\nParseval check: Σ f̂(S)² =", np.sum(f_hat**2), "(should be 1.0)")
print("Energy at degree 1:", 3 * (1/2)**2, "= 75%")
print("Energy at degree 3:", (1/2)**2, "= 25%")

In [ ]:
# ── Parseval's identity for several functions ─────────────────────────
print("Parseval check (Σ f̂(S)² should equal E[f²] = 1 for Boolean fns):\n")
for name, f_vals in [("AND_2", make_and(2)), ("Maj_3", make_majority(3)),
                      ("Parity_3", make_parity(3)), ("Dict_3", make_dictator(3))]:
    fh = fourier_coefficients(f_vals)
    energy = np.sum(fh**2)
    empirical = np.mean(f_vals**2)
    print(f"  {name:>10s}: Σ f̂² = {energy:.6f},  E[f²] = {empirical:.6f}")

In [ ]:
# ── Degree-k truncation and sign-rounding (Example 2.7 / Prop 2.8) ───
n = 3
f = make_majority(n)
f_hat = fourier_coefficients(f)
N = 1 << n

# Truncate to degree ≤ 1: zero out coefficients with |S| > 1
f_hat_trunc = f_hat.copy()
for r in range(N):
    S = index_to_subset(r, n)
    if len(S) > 1:
        f_hat_trunc[r] = 0.0

# Reconstruct continuous approximation
f_approx = evaluate_from_coefficients(f_hat_trunc)

# Sign-round
f_rounded = np.sign(f_approx)
# Handle zeros (sgn(0) is undefined; assign +1 as per convention)
f_rounded[f_rounded == 0] = 1.0

print("Degree-≤1 truncation of Maj_3:")
print(f"  Truncated coefficients: {f_hat_trunc}")
print(f"  Continuous approx f^{{≤1}}(x): {f_approx}")
print(f"  sgn(f^{{≤1}}):                {f_rounded}")
print(f"  Original Maj_3:              {f}")
print(f"  Agreement: {np.mean(f_rounded == f)*100:.0f}%")
print(f"\n  Tail energy bound: Σ_{{|S|>1}} f̂(S)² = {np.sum(f_hat**2) - np.sum(f_hat_trunc**2):.4f}")

In [ ]:
# ── Convolution theorem (Theorem 2.11) ────────────────────────────────
# Dyadic convolution: (f * g)(x) = E_y[f(y) g(x ⊕ y)]
# where ⊕ is coordinatewise multiplication in {±1}^n.
# In spectral domain: (f*g)^(S) = f̂(S) · ĝ(S).

n = 3
f = make_majority(n)
g = make_parity(n)
N = 1 << n

f_hat = fourier_coefficients(f)
g_hat = fourier_coefficients(g)

# Compute convolution in spatial domain (brute force)
pts = hypercube_points(n)
conv_spatial = np.zeros(N)
for c in range(N):
    # x ⊕ y in {±1} convention is coordinatewise multiplication
    total = 0.0
    for d in range(N):
        # y = pts[d], x = pts[c], x⊕y has index c XOR d
        total += f[d] * g[c ^ d]  # bitwise XOR on indices = ⊕ on {±1}
    conv_spatial[c] = total / N

# Compute convolution in spectral domain
conv_spectral_hat = f_hat * g_hat
conv_spectral = evaluate_from_coefficients(conv_spectral_hat)

print("Convolution theorem check:")
print(f"  Spatial domain:  {conv_spatial}")
print(f"  Spectral domain: {conv_spectral}")
print(f"  Match: {np.allclose(conv_spatial, conv_spectral)}")

---
## §2. Ch3 — Influence and Noise

Compute influences via both the combinatorial definition and the spectral
formula. Apply the noise operator and verify noise stability.

In [ ]:
# ── Influence: combinatorial vs spectral (Theorem 3.1) ────────────────

def influence_combinatorial(f_values: np.ndarray, n: int, i: int) -> float:
    """Inf_i[f] = Pr[f(x) ≠ f(x^{⊕i})] where x^{⊕i} flips coordinate i."""
    N = 1 << n
    bit = 1 << (i - 1)  # bit position for coordinate i
    count = sum(1 for c in range(N) if f_values[c] != f_values[c ^ bit])
    return count / N


def influence_spectral(f_hat: np.ndarray, n: int, i: int) -> float:
    """Inf_i[f] = Σ_{S ∋ i} f̂(S)² (spectral formula)."""
    N = 1 << n
    total = 0.0
    for r in range(N):
        S = index_to_subset(r, n)
        if i in S:
            total += f_hat[r] ** 2
    return total


# Verify for Maj_3 (Example 3.2: each influence = 1/2)
n = 3
f = make_majority(n)
f_hat = fourier_coefficients(f)

print("Maj_3 influences:")
for i in range(1, n + 1):
    ic = influence_combinatorial(f, n, i)
    isp = influence_spectral(f_hat, n, i)
    print(f"  Inf_{i}:  combinatorial = {ic:.4f},  spectral = {isp:.4f}")

total_inf = sum(influence_spectral(f_hat, n, i) for i in range(1, n + 1))
print(f"\nTotal influence I[Maj_3] = {total_inf:.4f} (expected: 3/2 = 1.5)")

# Also verify: I[f] = Σ_S |S| · f̂(S)²
total_inf_alt = sum(len(index_to_subset(r, n)) * f_hat[r]**2 for r in range(1 << n))
print(f"Via Σ |S|·f̂(S)² = {total_inf_alt:.4f}")

In [ ]:
# ── Noise operator T_ρ (Definition 3.4, Example 3.5) ─────────────────
# Spectral definition: (T_ρ f)^(S) = ρ^|S| · f̂(S)

def noise_operator(f_hat: np.ndarray, rho: float, n: int) -> np.ndarray:
    """Apply T_ρ in spectral domain.  Returns the function values of T_ρ f."""
    N = 1 << n
    noised_hat = np.zeros(N)
    for r in range(N):
        k = bin(r).count('1')  # |S| = popcount of r
        noised_hat[r] = (rho ** k) * f_hat[r]
    return evaluate_from_coefficients(noised_hat)


# Example 3.5: T_{1/2} Maj_3
n = 3
f_hat = fourier_coefficients(make_majority(n))
T_half_maj = noise_operator(f_hat, 0.5, n)

print("T_{1/2} Maj_3 values (should be fractions of form k/16):")
pts = hypercube_points(n)
for c in range(1 << n):
    x = tuple(pts[c])
    print(f"  x = {x}:  (T_{{1/2}} Maj_3)(x) = {T_half_maj[c]:+.4f}")

# Verify against probabilistic computation:
# (T_{1/2} f)(x) = E_y[f(y)] where y is ρ-correlated with x
print("\nProbabilistic verification (sampling ρ-correlated pairs):")
rng = np.random.default_rng(42)
n_samples = 200_000
for c in [0, 1]:  # check a couple of points
    x = pts[c]
    agrees = 0
    for _ in range(n_samples):
        # Each y_i = x_i with prob (1+ρ)/2, else -x_i
        flip = rng.random(n) < 0.25  # prob of flip = (1-ρ)/2 = 0.25
        y = x.copy()
        y[flip] = -y[flip]
        agrees += np.sign(y.sum())  # Maj_3(y)
    empirical = agrees / n_samples
    print(f"  x = {tuple(x)}: spectral = {T_half_maj[c]:+.4f}, empirical ≈ {empirical:+.4f}")

In [ ]:
# ── Noise stability (Definition 3.6) ──────────────────────────────────
# Stab_ρ[f] = Σ_S ρ^|S| f̂(S)² = E[f(x)f(y)] for ρ-correlated (x,y)

def noise_stability(f_hat: np.ndarray, rho: float, n: int) -> float:
    N = 1 << n
    return sum((rho ** bin(r).count('1')) * f_hat[r]**2 for r in range(N))


n = 3
f_hat = fourier_coefficients(make_majority(n))

print("Noise stability of Maj_3:")
for rho in [0.0, 0.25, 0.5, 0.75, 1.0]:
    stab = noise_stability(f_hat, rho, n)
    print(f"  Stab_{{{rho:.2f}}} = {stab:.6f}")

# At ρ=1: Stab = Σ f̂(S)² = 1 (Parseval)
# At ρ=0: Stab = f̂(∅)² = 0 (since Maj is balanced)
print(f"\nStab_{{1/2}}[Maj_3] = {noise_stability(f_hat, 0.5, n):.6f}")
print(f"  Expected: 3·(1/2)²·(1/2) + (1/2)²·(1/2)³ = 3/8 + 1/16 = {3/8 + 1/16:.6f}")

---
## §3. Ch4 — Hypercontractivity and the Level-k Inequality

Numerical verification of the level-k bound and the LMN theorem.

In [ ]:
# ── Level-k inequality (Theorem 4.3): W^k[f] ≤ (e·I[f]/k)^k ─────────

def spectral_weight_at_degree(f_hat: np.ndarray, n: int, k: int) -> float:
    """W^k[f] = Σ_{|S|=k} f̂(S)²."""
    N = 1 << n
    return sum(f_hat[r]**2 for r in range(N) if bin(r).count('1') == k)


def total_influence_spectral(f_hat: np.ndarray, n: int) -> float:
    """I[f] = Σ_S |S| · f̂(S)²."""
    N = 1 << n
    return sum(bin(r).count('1') * f_hat[r]**2 for r in range(N))


from math import e as E

# --- Maj_3 ---
n = 3
f_hat = fourier_coefficients(make_majority(n))
I_f = total_influence_spectral(f_hat, n)

print(f"Maj_3: I[f] = {I_f:.4f}")
print(f"{'k':>3}  {'W^k (actual)':>14}  {'(eI/k)^k (bound)':>18}  {'tight?':>8}")
for k in range(1, n + 1):
    wk = spectral_weight_at_degree(f_hat, n, k)
    bound = (E * I_f / k) ** k
    print(f"{k:3d}  {wk:14.6f}  {bound:18.6f}  {'vacuous' if bound > 1 else ''}")

# --- AND_10 ---
print()
n = 10
f_hat = fourier_coefficients(make_and(n))
I_f = total_influence_spectral(f_hat, n)

print(f"AND_{{10}}: I[f] = {I_f:.6f}")
print(f"{'k':>3}  {'W^k (actual)':>14}  {'(eI/k)^k (bound)':>18}")
for k in range(1, 6):
    wk = spectral_weight_at_degree(f_hat, n, k)
    bound = (E * I_f / k) ** k
    print(f"{k:3d}  {wk:14.6f}  {bound:18.6f}")

In [ ]:
# ── LMN theorem bound (Theorem 4.5) ──────────────────────────────────
# For AC⁰ circuits of depth d and size M on n inputs:
#   Σ_{|S|≥k} f̂(S)² ≤ 2M · 2^{-k^{1/(d-1)}/20}
# k* = (20 ln(2M/ε))^{d-1}

from math import log, ceil

def lmn_k_star(d: int, M: int, eps: float) -> float:
    """Compute k* such that tail energy ≤ ε for AC⁰ depth d, size M."""
    return (20 * log(2 * M / eps)) ** (d - 1)


# Example from the guide: depth 3, size 10000, ε = 0.01
d, M, eps = 3, 10_000, 0.01
k = lmn_k_star(d, M, eps)
print(f"LMN theorem: depth={d}, size={M}, ε={eps}")
print(f"  k* = (20·ln(2·{M}/{eps}))^{{{d-1}}} = {k:.1f}")
print(f"  So degree-{ceil(k)} truncation captures ≥ {(1-eps)*100:.0f}% of energy")

# Sweep over depths
print(f"\nk* for M={M}, ε={eps}, varying depth:")
for d in range(2, 7):
    k = lmn_k_star(d, M, eps)
    print(f"  depth {d}: k* = {k:.1f}")

---
## §4. Ch5 — The Walsh–Hadamard Transform

Step-by-step FWHT trace, matrix construction, and timing comparison.

In [ ]:
# ── Hadamard matrix construction (Definition 5.1) ────────────────────

def hadamard_matrix(n: int) -> np.ndarray:
    """Build H_n via Kronecker product: H_n = H_1 ⊗ H_{n-1}."""
    H = np.array([[1]])
    H1 = np.array([[1, 1], [1, -1]])
    for _ in range(n):
        H = np.kron(H1, H)
    return H


# Display H_2
H2 = hadamard_matrix(2)
print("H_2 =")
print(H2)

# Verify: H_n² = N·I (Theorem 5.1(iv))
n = 4
Hn = hadamard_matrix(n)
N = 1 << n
assert np.allclose(Hn @ Hn, N * np.eye(N))
print(f"\n✓ H_{n}² = {N}·I_{N}")

In [ ]:
# ── FWHT step-by-step trace (Example 5.3: n=2) ──────────────────────
def fwht_traced(v: np.ndarray) -> list:
    """FWHT with intermediate states printed."""
    a = v.astype(float).copy()
    N = len(a)
    states = [a.copy()]
    h = 1
    ell = 0
    while h < N:
        for j in range(0, N, 2 * h):
            for i in range(j, j + h):
                x, y = a[i], a[i + h]
                a[i], a[i + h] = x + y, x - y
        states.append(a.copy())
        ell += 1
        h *= 2
    return states


# Example 5.3: v = (1, -1, -1, 1) = x_1 x_2
v = np.array([1, -1, -1, 1])
states = fwht_traced(v)
print("FWHT trace for v = (1, -1, -1, 1):")
print(f"  Initial:   {states[0]}")
for i in range(1, len(states)):
    print(f"  After ℓ={i-1}: {states[i]}")

# Verify against direct matrix multiplication
H2 = hadamard_matrix(2)
direct = H2 @ v
print(f"\n  Direct H_2·v = {direct}")
print(f"  Match: {np.allclose(states[-1], direct)}")

# n=3 example from ch7: parity function
print("\n--- n=3: parity function ---")
v3 = np.array([1, -1, -1, 1, -1, 1, 1, -1])
states3 = fwht_traced(v3)
print(f"  Initial:   {states3[0]}")
for i in range(1, len(states3)):
    print(f"  After ℓ={i-1}: {states3[i]}")
print(f"  Normalized: {states3[-1] / 8}")

In [ ]:
# ── Timing: FWHT vs naive matrix multiply ─────────────────────────────
import time

def time_fwht(n, impl='numpy', repeats=5):
    N = 1 << n
    v = np.random.choice([-1, 1], size=N).astype(float)
    if impl == 'numpy':
        fn = lambda: fwht_numpy(v)
    elif impl == 'torch':
        vt = torch.tensor(v).unsqueeze(0)
        fn = lambda: fwht_torch(vt)
    else:  # naive
        H = hadamard_matrix(n)
        fn = lambda: H @ v
    times = []
    for _ in range(repeats):
        t0 = time.perf_counter()
        fn()
        times.append(time.perf_counter() - t0)
    return min(times)


print(f"{'n':>3} {'N':>7} {'FWHT (np)':>12} {'FWHT (torch)':>14} {'Naive H·v':>12}")
for n in range(4, 17, 2):
    N = 1 << n
    t_np = time_fwht(n, 'numpy')
    t_torch = time_fwht(n, 'torch')
    t_naive = time_fwht(n, 'naive') if n <= 12 else float('nan')
    print(f"{n:3d} {N:7d} {t_np:12.6f}s {t_torch:14.6f}s {t_naive:12.6f}s")

---
## §5. Ch6 — Reed–Muller Codes and the Degree-Increase Phenomenon

Construct RM codewords, verify the degree increase under $v \mapsto (-1)^v$,
and compute Chow parameters of LTFs.

In [ ]:
# ── Construct RM(1,2) codewords (Example 6.3) ─────────────────────────
# RM(1,2): evaluation vectors of all affine GF(2)-functions in 2 variables.
# Affine functions: c_0 ⊕ c_1·b_1 ⊕ c_2·b_2 for c_i ∈ {0,1}.

n = 2
N = 1 << n  # 4

# All points of {0,1}^2 in binary-counting order
gf2_points = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])  # but we use (b1,b2) with b1=LSB

# Correct order matching our hypercube convention: index c -> (b_1, b_2) = (c&1, (c>>1)&1)
gf2_pts = np.array([[(c >> i) & 1 for i in range(n)] for c in range(N)])

print("RM(1,2) codewords — affine functions over GF(2):\n")
print(f"{'Function':>20s}  {'GF(2) eval':>14s}  {'±1 codeword':>20s}  {'Walsh deg':>10s}")
print("-" * 70)

for c0 in range(2):
    for c1 in range(2):
        for c2 in range(2):
            # Evaluate c0 ⊕ c1·b1 ⊕ c2·b2 at all 4 points
            gf2_eval = np.array([(c0 ^ (c1 * b[0]) ^ (c2 * b[1])) for b in gf2_pts])
            # Map to ±1: v -> (-1)^v
            pm1_cw = (-1) ** gf2_eval

            # Compute Walsh degree of ±1 codeword
            fh = fourier_coefficients(pm1_cw)
            max_deg = max((bin(r).count('1') for r in range(N) if abs(fh[r]) > 1e-10), default=0)

            func_str = f"{c0}" + (f" ⊕ b₁" if c1 else "") + (f" ⊕ b₂" if c2 else "")
            if not c1 and not c2:
                func_str = str(c0)
            print(f"{func_str:>20s}  {str(tuple(gf2_eval)):>14s}  {str(tuple(pm1_cw)):>20s}  {max_deg:>10d}")

print("\nNote: b₁⊕b₂ has GF(2)-degree 1 but its ±1 codeword (+1,-1,-1,+1) = x₁x₂")
print("has Walsh degree 2. This is the degree-increase phenomenon.")

In [ ]:
# ── Degree increase: ⊕_i b_i → full parity (Walsh degree n) ──────────
# GF(2) parity ⊕_i b_i has degree 1 over GF(2), but its ±1 image
# (-1)^{b_1⊕...⊕b_n} = (-1)^{b_1}·...·(-1)^{b_n} = x_1·...·x_n = χ_{[n]}
# has Walsh degree n.

print("Degree increase: GF(2)-parity → Walsh parity\n")
for n in range(2, 7):
    N = 1 << n
    # GF(2) parity: ⊕_i b_i evaluated at all {0,1}^n points
    gf2_pts = np.array([[(c >> i) & 1 for i in range(n)] for c in range(N)])
    gf2_parity = gf2_pts.sum(axis=1) % 2  # XOR = sum mod 2

    # Map to ±1
    pm1_parity = (-1.0) ** gf2_parity

    # This should be the full parity function χ_{[n]} = x_1·...·x_n
    expected = make_parity(n)
    assert np.allclose(pm1_parity, expected), f"Failed for n={n}"

    # Verify Walsh degree = n (only coefficient at S=[n])
    fh = fourier_coefficients(pm1_parity)
    max_idx = np.argmax(np.abs(fh))
    assert max_idx == N - 1, f"Expected index {N-1}, got {max_idx}"
    assert np.allclose(np.abs(fh[max_idx]), 1.0)

    print(f"  n={n}: GF(2)-degree 1 → Walsh degree {n}  "
          f"(f̂({{1,...,{n}}}) = {fh[N-1]:+.0f}, all others ≈ 0)  ✓")

In [ ]:
# ── Chow parameters of LTFs (Theorem 6.1) ────────────────────────────
# Chow parameters: (f̂(∅), f̂({1}), ..., f̂({n}))
# For LTFs, these uniquely determine the function (Chow's theorem).

def chow_parameters(f_values: np.ndarray, n: int) -> np.ndarray:
    """Compute the n+1 Chow parameters of f."""
    fh = fourier_coefficients(f_values)
    # f̂(∅) is at index 0; f̂({i}) is at index 2^(i-1)
    chow = np.zeros(n + 1)
    chow[0] = fh[0]  # f̂(∅)
    for i in range(1, n + 1):
        chow[i] = fh[1 << (i - 1)]  # f̂({i})
    return chow


print("Chow parameters of various LTFs:\n")

# Majority on 3, 5, 7 variables
for n in [3, 5, 7]:
    f = make_majority(n)
    chow = chow_parameters(f, n)
    print(f"Maj_{n}:")
    print(f"  f̂(∅) = {chow[0]:+.6f}")
    for i in range(1, n + 1):
        print(f"  f̂({{{i}}}) = {chow[i]:+.6f}")
    # All degree-1 coefficients should be equal by symmetry
    assert np.allclose(chow[1:], chow[1]), f"Symmetry broken for Maj_{n}"
    print(f"  (all degree-1 coefficients equal = {chow[1]:.6f})")
    # Exact formula: f̂({i}) = C(n-1, (n-1)/2) / 2^(n-1)
    exact = comb(n - 1, (n - 1) // 2) / (2 ** (n - 1))
    print(f"  Exact: C({n-1},{(n-1)//2}) / 2^{n-1} = {exact:.6f}")
    print()

# Weighted threshold: f(x) = sgn(3x_1 + x_2 + x_3)
n = 3
pts = hypercube_points(n)
f_weighted = np.sign(3 * pts[:, 0] + pts[:, 1] + pts[:, 2]).astype(float)
# Handle zeros
f_weighted[f_weighted == 0] = 1.0
chow_w = chow_parameters(f_weighted, n)
print(f"sgn(3x₁ + x₂ + x₃):")
print(f"  Chow = ({', '.join(f'{c:+.4f}' for c in chow_w)})")
print(f"  Note: f̂({{1}}) > f̂({{2}}) = f̂({{3}}), reflecting the larger weight on x₁")

---
## §6. Ch7 — Spectral Parameterization Pipeline

End-to-end demonstration of the thesis's pipeline:
binary weight vector → spectrum → truncate → reconstruct → sign → error.
Also: parameter count comparison and CLT normalization histogram.

In [ ]:
# ── End-to-end pipeline (Section 7.5 worked example) ──────────────────
# Trace the full spectral parameterization for a single row.

def spectral_pipeline(w: np.ndarray, k: int, verbose: bool = True):
    """Run the full spectral parameterization pipeline on weight vector w.

    Steps: compute spectrum → truncate to degree ≤ k → reconstruct → sign.
    Returns (w_reconstructed, error_rate, n_params, n_weights).
    """
    N = len(w)
    n = int(np.log2(N))
    assert N == 1 << n

    # Step 1: Fourier coefficients
    f_hat = fourier_coefficients(w)

    # Step 2: Truncate to degree ≤ k
    f_hat_trunc = np.zeros(N)
    n_params = 0
    for r in range(N):
        deg = bin(r).count('1')
        if deg <= k:
            f_hat_trunc[r] = f_hat[r]
            n_params += 1

    # Step 3: Reconstruct continuous weights
    z = evaluate_from_coefficients(f_hat_trunc)

    # Step 4: Sign-round
    w_recon = np.sign(z)
    w_recon[w_recon == 0] = 1.0  # convention for ties

    # Error rate
    error_rate = np.mean(w_recon != w)

    # Tail energy (upper bound on error)
    tail_energy = np.sum(f_hat**2) - np.sum(f_hat_trunc**2)

    if verbose:
        print(f"  n = {n}, N = {N}")
        print(f"  Truncation degree k = {k}")
        print(f"  Parameters: {n_params} (vs {N} weights) — {N/n_params:.1f}× compression")
        print(f"  Tail energy Σ_{{|S|>k}} f̂(S)² = {tail_energy:.6f}")
        print(f"  Sign-rounding error: {error_rate*100:.1f}% ({int(error_rate*N)}/{N} mismatches)")

    return w_recon, error_rate, n_params, N


# --- Example 1: Majority (spectrally concentrated) ---
print("=" * 60)
print("Pipeline for Maj_3 (spectrally concentrated):")
print("=" * 60)
w = make_majority(3)
spectral_pipeline(w, k=1)

# --- Example 2: Parity (spectrally dispersed) ---
print(f"\n{'=' * 60}")
print("Pipeline for Parity_3 (spectrally dispersed — worst case):")
print("=" * 60)
w = make_parity(3)
spectral_pipeline(w, k=1)

# --- Example 3: Random Boolean function at n=10 ---
print(f"\n{'=' * 60}")
print("Pipeline for random Boolean function (n=10, d=1024):")
print("=" * 60)
rng = np.random.default_rng(42)
n = 10
w = rng.choice([-1, 1], size=1 << n).astype(float)
for k in [1, 2, 3]:
    print(f"\n  --- k = {k} ---")
    spectral_pipeline(w, k)

In [ ]:
# ── Parameter count comparison (Section 7.3) ─────────────────────────
# C(n, ≤k) = Σ_{i=0}^{k} C(n,i) vs N = 2^n

print("Parameter count: C(n,≤k) vs N=2^n\n")
print(f"{'n':>4}  {'d=2^n':>7}  {'k=1':>7}  {'k=2':>7}  {'k=3':>7}  {'k=4':>7}")
print("-" * 50)
for n in [4, 6, 8, 10, 12, 14]:
    N = 1 << n
    counts = []
    for k in [1, 2, 3, 4]:
        c = sum(comb(n, i) for i in range(k + 1))
        counts.append(c)
    print(f"{n:4d}  {N:7d}  {counts[0]:7d}  {counts[1]:7d}  {counts[2]:7d}  {counts[3]:7d}")

print(f"\nThesis setting: n=10, k=3 → {sum(comb(10,i) for i in range(4))} params "
      f"vs 1024 weights = {1024/176:.1f}× compression")

In [ ]:
# ── CLT normalization histogram (Section 7.4) ────────────────────────
# For W ∈ {±1}^{d×d} and uniform x ∈ {±1}^d, each output y_j = w_j^T x
# is a sum of d independent ±1 r.v.s. By CLT, y_j / √d → N(0,1).

from scipy.stats import norm

d = 1024
n_rows = 100
n_samples = 5000
rng = np.random.default_rng(123)

# Generate random binary weight matrix and inputs
W = rng.choice([-1, 1], size=(n_rows, d)).astype(float)

# Collect normalized outputs
normalized_outputs = []
for _ in range(n_samples):
    x = rng.choice([-1, 1], size=d).astype(float)
    y = W @ x  # (n_rows,)
    normalized_outputs.append(y / np.sqrt(d))

normalized_outputs = np.array(normalized_outputs).flatten()

# Plot histogram vs N(0,1)
fig, ax = plt.subplots(1, 1, figsize=(8, 4))
ax.hist(normalized_outputs, bins=80, density=True, alpha=0.7, color='steelblue',
        label=f'$w^T x / \\sqrt{{d}}$  (d={d})')
zz = np.linspace(-4, 4, 200)
ax.plot(zz, norm.pdf(zz), 'r-', lw=2, label='$\\mathcal{N}(0,1)$')
ax.set_xlabel('$w^T x / \\sqrt{d}$')
ax.set_ylabel('Density')
ax.set_title(f'CLT normalization: {n_rows} rows × {n_samples} samples, d={d}')
ax.legend()
ax.set_xlim(-4, 4)
plt.tight_layout()
plt.show()

# Also print summary statistics
print(f"\nSummary: mean = {normalized_outputs.mean():.4f} (expect 0)")
print(f"         var  = {normalized_outputs.var():.4f} (expect 1)")
print(f"         1/√d = 1/√{d} = {1/np.sqrt(d):.6f} = 2^{{-{int(np.log2(np.sqrt(d)))}}}")

In [ ]:
# ── PyTorch: batch spectral pipeline ──────────────────────────────────
# Batch version: process an entire weight matrix at once (GPU-friendly).

def spectral_pipeline_torch(W: torch.Tensor, n: int, k: int):
    """Spectral parameterization pipeline for a batch of weight rows.

    W: (num_rows, 2^n) tensor of ±1 weights.
    Returns (W_reconstructed, error_rate).
    """
    N = 1 << n
    num_rows = W.shape[0]

    # Step 1: FWHT (batch) → normalize
    F_hat = fwht_torch(W.unsqueeze(0) if W.dim() == 1 else W.unsqueeze(0))
    F_hat = F_hat.squeeze(0) / N  # (num_rows, N)

    # Step 2: Build degree mask — keep only |S| ≤ k
    degrees = torch.tensor([bin(r).count('1') for r in range(N)])
    mask = (degrees <= k).float()  # (N,)
    F_hat_trunc = F_hat * mask.unsqueeze(0)

    # Step 3: Reconstruct via FWHT (inverse = same matrix)
    Z = fwht_torch(F_hat_trunc.unsqueeze(0)).squeeze(0)

    # Step 4: Sign
    W_recon = torch.sign(Z)
    W_recon[W_recon == 0] = 1.0

    error_rate = (W_recon != W).float().mean().item()
    n_params = mask.sum().int().item()

    return W_recon, error_rate, n_params


# Demo: batch pipeline on 16 random rows, n=10
n = 10
N = 1 << n
torch.manual_seed(42)
W = (2 * torch.randint(0, 2, (16, N)) - 1).float()

print("PyTorch batch spectral pipeline (16 rows, n=10):\n")
for k in [1, 2, 3, 4]:
    W_rec, err, n_params = spectral_pipeline_torch(W, n, k)
    print(f"  k={k}: {n_params:4d} params, error = {err*100:.1f}%, "
          f"compression = {N/n_params:.1f}×")